# Lesson 01 ? Python AI Mindset

Welcome! In this lesson we will warm up with Python tools that power modern AI systems. By the end you will have trained a tiny classifier from scratch and visualized how it learns.

> **Focus:** Thinking like an AI builder and translating intuition into code.

## Learning Goals
- Understand what it means to model intelligence with code
- Generate and inspect a synthetic dataset for a classification task
- Implement gradient descent to fit a logistic regression model
- Visualize the trained model's predictions

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use("seaborn-v0_8")

## Step 1 ? Build an Intuition Playground
We'll craft a toy dataset where each sample is a 2D point. Points near the center belong to class `1` while points farther out belong to class `0`. This gives us a smooth decision boundary that our logistic model must learn.

In [ ]:
def make_ring_dataset(n_samples: int = 400, radius: float = 1.2):
    angles = np.random.uniform(0, 2 * np.pi, size=n_samples)
    distances = np.random.normal(loc=radius, scale=0.35, size=n_samples)

    x_outer = np.c_[
        (distances * np.cos(angles)),
        (distances * np.sin(angles))
    ]
    y_outer = np.zeros(n_samples)

    x_inner = np.random.normal(loc=0.0, scale=0.45, size=(n_samples, 2))
    y_inner = np.ones(n_samples)

    X = np.vstack([x_inner, x_outer])
    y = np.concatenate([y_inner, y_outer])

    # Shuffle to avoid any ordering bias
    shuffle_idx = np.random.permutation(len(X))
    return X[shuffle_idx], y[shuffle_idx]


X, y = make_ring_dataset()
X[:5], y[:5]

Let's visualize the dataset. You should see a glowing ring?the perfect target for a logistic model to split.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", alpha=0.75, edgecolor="white", s=60)
ax.set_title("Synthetic Ring Dataset")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_aspect("equal", adjustable="box")
ax.legend(*scatter.legend_elements(), title="Class")
plt.show()

## Step 2 ? Logistic Regression From Scratch
Instead of calling a library, we'll implement a compact logistic regression and train it with gradient descent. This is the same core idea inside more advanced AI models?stack enough of these neurons and you have a deep network.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def predict_proba(weights, bias, inputs):
    return sigmoid(inputs @ weights + bias)


def compute_loss(weights, bias, inputs, targets):
    probs = predict_proba(weights, bias, inputs)
    eps = 1e-8
    return -np.mean(targets * np.log(probs + eps) + (1 - targets) * np.log(1 - probs + eps))

In [ ]:
def train_logistic_regression(inputs, targets, lr=0.2, epochs=800):
    n_samples, n_features = inputs.shape
    weights = np.random.randn(n_features) * 0.1
    bias = 0.0
    history = []

    for epoch in range(1, epochs + 1):
        probs = predict_proba(weights, bias, inputs)
        error = probs - targets

        grad_w = inputs.T @ error / n_samples
        grad_b = np.mean(error)

        weights -= lr * grad_w
        bias -= lr * grad_b

        if epoch % 50 == 0 or epoch == 1:
            loss = compute_loss(weights, bias, inputs, targets)
            history.append((epoch, float(loss)))

    return weights, bias, history


weights, bias, history = train_logistic_regression(X, y)
weights, bias, history[:5]

### Loss Curve Peek
The loss should decay as the model finds a decision boundary. Let's inspect it.

In [ ]:
epochs, losses = zip(*history)
plt.figure(figsize=(6, 4))
plt.plot(epochs, losses, marker="o")
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy")
plt.grid(True)
plt.show()

### Visualize the Decision Boundary
We'll classify each grid point to get a heatmap, layered with the training data.

In [ ]:
grid_lin = np.linspace(-2.5, 2.5, 200)
xx, yy = np.meshgrid(grid_lin, grid_lin)
flat_grid = np.c_[xx.ravel(), yy.ravel()]
probs_grid = predict_proba(weights, bias, flat_grid).reshape(xx.shape)

plt.figure(figsize=(6, 6))
contour = plt.contourf(xx, yy, probs_grid, levels=20, cmap="coolwarm", alpha=0.8)
plt.colorbar(contour, label="Predicted Probability of Class 1")
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="black", s=40)
plt.title("Decision Boundary Learned From Scratch")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()


In [ ]:
preds = (predict_proba(weights, bias, X) >= 0.5).astype(int)
accuracy = (preds == y).mean()
print(f"Training accuracy: {accuracy:.3f}")

## Wrap-Up
With less than 50 lines of core logic we reproduced the mathematics that drives binary classification in many production AI systems. Experiment by changing the learning rate, the number of epochs, or by perturbing the dataset. Notice how the decision boundary morphs?this is the beginning of mastering model intuition.